<a href="https://colab.research.google.com/github/gianni04/projet-google-colab/blob/main/option.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go
import matplotlib.pyplot as plt

In [ ]:
class VanillaOption:
    """Représente le contrat de l'option (Statique)."""
    def __init__(self, strike, expiry_years, option_type='Call', is_american=False):
        self.K = strike
        self.T = expiry_years
        self.option_type = option_type.capitalize()
        self.is_american = is_american

        if self.option_type not in ['Call', 'Put']:
            raise ValueError("Type d'option invalide. Utilisez 'Call' ou 'Put'.")

class MarketEnvironment:
    """Représente les conditions de marché (Dynamique)."""
    def __init__(self, spot, rate, dividend_yield, volatility):
        self.S = spot
        self.r = rate
        self.q = dividend_yield
        self.sigma = volatility

In [ ]:
class BlackScholesPricer:
    """Moteur de pricing analytique pour options Européennes."""
    def __init__(self, option: VanillaOption, market: MarketEnvironment):
        if option.is_american:
            raise TypeError("Black-Scholes ne peut pas pricer d'options Américaines proprement.")
        self.opt = option
        self.mkt = market

    def _d1_d2(self):
        d1 = (np.log(self.mkt.S / self.opt.K) + (self.mkt.r - self.mkt.q + 0.5 * self.mkt.sigma**2) * self.opt.T) / (self.mkt.sigma * np.sqrt(self.opt.T))
        d2 = d1 - self.mkt.sigma * np.sqrt(self.opt.T)
        return d1, d2

    def price(self):
        d1, d2 = self._d1_d2()
        if self.opt.option_type == 'Call':
            return self.mkt.S * np.exp(-self.mkt.q * self.opt.T) * stats.norm.cdf(d1) - self.opt.K * np.exp(-self.mkt.r * self.opt.T) * stats.norm.cdf(d2)
        else:
            return self.opt.K * np.exp(-self.mkt.r * self.opt.T) * stats.norm.cdf(-d2) - self.mkt.S * np.exp(-self.mkt.q * self.opt.T) * stats.norm.cdf(-d1)

    def greeks(self):
        """Retourne les Grecques de premier ordre (Delta, Gamma, Vega, Theta)."""
        d1, d2 = self._d1_d2()
        pdf_d1 = stats.norm.pdf(d1)

        if self.opt.option_type == 'Call':
            delta = np.exp(-self.mkt.q * self.opt.T) * stats.norm.cdf(d1)
        else:
            delta = np.exp(-self.mkt.q * self.opt.T) * (stats.norm.cdf(d1) - 1)

        gamma = (np.exp(-self.mkt.q * self.opt.T) * pdf_d1) / (self.mkt.S * self.mkt.sigma * np.sqrt(self.opt.T))

        vega = (self.mkt.S * np.exp(-self.mkt.q * self.opt.T) * pdf_d1 * np.sqrt(self.opt.T)) / 100

        term1 = -(self.mkt.S * self.mkt.sigma * np.exp(-self.mkt.q * self.opt.T) * pdf_d1) / (2 * np.sqrt(self.opt.T))
        if self.opt.option_type == 'Call':
            term2 = self.mkt.q * self.mkt.S * np.exp(-self.mkt.q * self.opt.T) * stats.norm.cdf(d1)
            term3 = self.mkt.r * self.opt.K * np.exp(-self.mkt.r * self.opt.T) * stats.norm.cdf(d2)
            theta = (term1 + term2 - term3) / 365
        else:
            term2 = self.mkt.q * self.mkt.S * np.exp(-self.mkt.q * self.opt.T) * stats.norm.cdf(-d1)
            term3 = self.mkt.r * self.opt.K * np.exp(-self.mkt.r * self.opt.T) * stats.norm.cdf(-d2)
            theta = (term1 - term2 + term3) / 365

        return {'Delta': delta, 'Gamma': gamma, 'Vega': vega, 'Theta': theta}

In [ ]:
class BinomialTreePricer:
    """Moteur de pricing par Arbre Binomial (Modèle CRR)."""
    def __init__(self, option: VanillaOption, market: MarketEnvironment, steps=500):
        self.opt = option
        self.mkt = market
        self.N = steps

    def price(self):
        dt = self.opt.T / self.N
        u = np.exp(self.mkt.sigma * np.sqrt(dt))
        d = 1 / u
        p = (np.exp((self.mkt.r - self.mkt.q) * dt) - d) / (u - d)
        discount = np.exp(-self.mkt.r * dt)

        ST = self.mkt.S * (u ** np.arange(self.N, -1, -1)) * (d ** np.arange(0, self.N + 1, 1))

        if self.opt.option_type == 'Call':
            V = np.maximum(0, ST - self.opt.K)
        else:
            V = np.maximum(0, self.opt.K - ST)

        for i in range(self.N - 1, -1, -1):
            ST = ST[:-1] / u
            V = discount * (p * V[:-1] + (1 - p) * V[1:])

            if self.opt.is_american:
                if self.opt.option_type == 'Call':
                    payoff = np.maximum(0, ST - self.opt.K)
                else:
                    payoff = np.maximum(0, self.opt.K - ST)
                V = np.maximum(V, payoff)

        return V[0]

In [ ]:
S, K, T, r, q, sigma = 100.0, 100.0, 1.0, 0.05, 0.02, 0.20
mkt = MarketEnvironment(S, r, q, sigma)

call_eur = VanillaOption(K, T, option_type='Call', is_american=False)
call_ame = VanillaOption(K, T, option_type='Call', is_american=True)

bsm_pricer = BlackScholesPricer(call_eur, mkt)
tree_eur = BinomialTreePricer(call_eur, mkt, steps=500)
tree_ame = BinomialTreePricer(call_ame, mkt, steps=500)

print("--- RÉSULTATS DU PRICING ---")
print(f"BSM Call Européen    : {bsm_pricer.price():.4f}")
print(f"Arbre Call Européen  : {tree_eur.price():.4f} (Converge vers BSM)")
print(f"Arbre Call Américain : {tree_ame.price():.4f} (Prime d'exercice anticipé)\n")

print("--- GRECQUES BSM (Call Eur) ---")
greeks = bsm_pricer.greeks()
for g, val in greeks.items():
    print(f"{g:<6}: {val:.4f}")

In [ ]:
from plotly.subplots import make_subplots

def plot_option_metrics():
    spots = np.linspace(50, 150, 200)
    K, T, r, q, sigma = 100.0, 1.0, 0.05, 0.02, 0.20

    prices, deltas, gammas, vegas, thetas = [], [], [], [], []

    for S in spots:
        mkt = MarketEnvironment(S, r, q, sigma)
        opt = VanillaOption(K, T, option_type='Call')
        pricer = BlackScholesPricer(opt, mkt)

        prices.append(pricer.price())
        g = pricer.greeks()
        deltas.append(g['Delta'])
        gammas.append(g['Gamma'])
        vegas.append(g['Vega'])
        thetas.append(g['Theta'])

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Prix du Call", "Delta (Couverture)", "Gamma (Risque de courbure)", "Vega (Sensibilité Volatilité)"),
        vertical_spacing=0.15
    )

    fig.add_trace(go.Scatter(x=spots, y=prices, name="Prix", line=dict(color="#00ffcc", width=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=spots, y=deltas, name="Delta", line=dict(color="#ff9900", width=3)), row=1, col=2)
    fig.add_trace(go.Scatter(x=spots, y=gammas, name="Gamma", line=dict(color="#ff3366", width=3)), row=2, col=1)
    fig.add_trace(go.Scatter(x=spots, y=vegas, name="Vega", line=dict(color="#33ccff", width=3)), row=2, col=2)

    for i in range(1, 3):
        for j in range(1, 3):
            fig.add_vline(x=K, line_dash="dash", line_color="white", opacity=0.3, row=i, col=j)

    fig.update_layout(
        title="<b>Profil de Risque : Call Option (Maturité 1 an)</b><br><sup>Ligne pointillée blanche = Strike (At-The-Money)</sup>",
        template="plotly_dark",
        height=700,
        showlegend=False,
        hovermode="x unified"
    )

    fig.update_xaxes(title_text="Prix du Sous-jacent (Spot)", row=2, col=1)
    fig.update_xaxes(title_text="Prix du Sous-jacent (Spot)", row=2, col=2)

    fig.show()

plot_option_metrics()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

style = {'description_width': 'initial'}
layout = widgets.Layout(width='350px')

w_type = widgets.ToggleButtons(options=['Call', 'Put'], description='Type:')
w_spot = widgets.FloatSlider(value=100.0, min=10.0, max=200.0, step=5.0, description='Centre Graphe:', style=style, layout=layout)
w_strike = widgets.FloatSlider(value=100.0, min=10.0, max=200.0, step=5.0, description='Strike (K):', style=style, layout=layout)
w_mat = widgets.FloatSlider(value=1.0, min=0.01, max=5.0, step=0.05, description='Maturité (Ans):', style=style, layout=layout)
w_vol = widgets.FloatSlider(value=0.20, min=0.05, max=1.0, step=0.01, description='Volatilité (σ):', style=style, layout=layout)
w_rate = widgets.FloatSlider(value=0.05, min=0.0, max=0.20, step=0.01, description='Taux (r):', style=style, layout=layout)

ui_left = widgets.VBox([w_type, w_spot, w_strike])
ui_right = widgets.VBox([w_mat, w_vol, w_rate])
ui = widgets.HBox([ui_left, ui_right])


out = widgets.Output()

def update_dashboard(change=None):
    with out:
        clear_output(wait=True)

        option_type = w_type.value
        spot_center = w_spot.value
        strike = w_strike.value
        maturity = w_mat.value
        vol = w_vol.value
        rate = w_rate.value
        div = 0.02

        spots = np.linspace(spot_center * 0.5, spot_center * 1.5, 100)
        prices, deltas, gammas, thetas = [], [], [], []

        for S in spots:
            mkt = MarketEnvironment(S, rate, div, vol)
            opt = VanillaOption(strike, maturity, option_type=option_type)
            pricer = BlackScholesPricer(opt, mkt)
            prices.append(pricer.price())
            g = pricer.greeks()
            deltas.append(g['Delta'])
            gammas.append(g['Gamma'])
            thetas.append(g['Theta'])

        fig = make_subplots(rows=2, cols=2, subplot_titles=(f"Prix {option_type}", "Delta", "Gamma", "Theta"))

        color = "#00ffcc" if option_type == 'Call' else "#ff5555"
        fig.add_trace(go.Scatter(x=spots, y=prices, line=dict(color=color, width=3)), row=1, col=1)
        fig.add_trace(go.Scatter(x=spots, y=deltas, line=dict(color="#ff9900", width=3)), row=1, col=2)
        fig.add_trace(go.Scatter(x=spots, y=gammas, line=dict(color="#ff3366", width=3)), row=2, col=1)
        fig.add_trace(go.Scatter(x=spots, y=thetas, line=dict(color="#b072e6", width=3)), row=2, col=2)

        for i in range(1, 3):
            for j in range(1, 3):
                fig.add_vline(x=strike, line_dash="dash", line_color="white", opacity=0.4, row=i, col=j)

        fig.update_layout(template="plotly_dark", height=600, showlegend=False, margin=dict(t=40, b=20, l=20, r=20))
        fig.show()

w_type.observe(update_dashboard, 'value')
w_spot.observe(update_dashboard, 'value')
w_strike.observe(update_dashboard, 'value')
w_mat.observe(update_dashboard, 'value')
w_vol.observe(update_dashboard, 'value')
w_rate.observe(update_dashboard, 'value')

display(ui, out)
update_dashboard()

In [ ]:
import numpy as np
import plotly.graph_objects as go

def plot_volatility_surface(S0=100.0):
    strikes = np.linspace(80, 120, 50)
    maturities = np.linspace(0.08, 2.0, 50)

    K, T = np.meshgrid(strikes, maturities)
    moneyness = K / S0

    base_vol = 0.15
    skew = -0.12 * (moneyness - 1.0)
    smile = 0.60 * (moneyness - 1.0)**2
    term_structure = 0.04 / np.sqrt(T)

    IV = base_vol + skew + smile + term_structure

    fig = go.Figure(data=[go.Surface(
        x=strikes,
        y=maturities,
        z=IV,
        colorscale='Viridis',
        contours=dict(
            z=dict(show=True, usecolormap=True, highlightcolor="limegreen", project_z=True)
        )
    )])

    fig.update_layout(
        title="<b>Surface de Volatilité Implicite (IV) 3D</b><br><sup>Moneyness Skew & Term Structure</sup>",
        scene=dict(
            xaxis_title='Strike (K)',
            yaxis_title='Maturité (T en années)',
            zaxis_title='Volatilité Implicite (σ)',
            camera=dict(eye=dict(x=1.6, y=-1.6, z=0.8))
        ),
        template="plotly_dark",
        height=800,
        margin=dict(l=0, r=0, b=0, t=60)
    )

    fig.show()

plot_volatility_surface(S0=100.0)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

style = {'description_width': 'initial'}
layout = widgets.Layout(width='350px')

w_type_am = widgets.ToggleButtons(options=['Call', 'Put'], description='Type:')
w_spot_am = widgets.FloatSlider(value=100.0, min=50.0, max=150.0, step=5.0, description='Centre Graphe:', style=style, layout=layout)
w_strike_am = widgets.FloatSlider(value=100.0, min=50.0, max=150.0, step=5.0, description='Strike (K):', style=style, layout=layout)
w_mat_am = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Maturité (Ans):', style=style, layout=layout)
w_vol_am = widgets.FloatSlider(value=0.25, min=0.05, max=0.80, step=0.01, description='Volatilité (σ):', style=style, layout=layout)
w_rate_am = widgets.FloatSlider(value=0.05, min=0.0, max=0.15, step=0.01, description='Taux (r):', style=style, layout=layout)
w_div_am = widgets.FloatSlider(value=0.0, min=0.0, max=0.15, step=0.01, description='Dividende (q):', style=style, layout=layout)
w_steps = widgets.IntSlider(value=50, min=10, max=200, step=10, description='Étapes Arbre (N):', style=style, layout=layout)

ui_left_am = widgets.VBox([w_type_am, w_spot_am, w_strike_am, w_steps])
ui_right_am = widgets.VBox([w_mat_am, w_vol_am, w_rate_am, w_div_am])
ui_am = widgets.HBox([ui_left_am, ui_right_am])

out_am = widgets.Output()

def update_american_dashboard(change=None):
    with out_am:
        clear_output(wait=True)

        opt_type = w_type_am.value
        spot_center = w_spot_am.value
        strike = w_strike_am.value
        T = w_mat_am.value
        vol = w_vol_am.value
        r = w_rate_am.value
        q = w_div_am.value
        N = w_steps.value

        spots = np.linspace(spot_center * 0.5, spot_center * 1.5, 40)
        prices_eur, prices_ame, premiums = [], [], []

        for S in spots:
            mkt = MarketEnvironment(S, r, q, vol)

            opt_eur = VanillaOption(strike, T, option_type=opt_type, is_american=False)
            pricer_eur = BlackScholesPricer(opt_eur, mkt)
            p_eur = pricer_eur.price()

            opt_ame = VanillaOption(strike, T, option_type=opt_type, is_american=True)
            pricer_ame = BinomialTreePricer(opt_ame, mkt, steps=N)
            p_ame = pricer_ame.price()

            prices_eur.append(p_eur)
            prices_ame.append(p_ame)
            premiums.append(max(0, p_ame - p_eur))

        fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Prix {opt_type} : Américain vs Européen", "Prime d'Exercice Anticipé"))

        fig.add_trace(go.Scatter(x=spots, y=prices_ame, name="Américain (Arbre)", line=dict(color="#00ffcc", width=3)), row=1, col=1)
        fig.add_trace(go.Scatter(x=spots, y=prices_eur, name="Européen (BSM)", line=dict(color="#ff5555", width=2, dash='dash')), row=1, col=1)

        fig.add_trace(go.Scatter(x=spots, y=premiums, name="Prime Anticipée", fill='tozeroy', line=dict(color="#ff9900", width=2)), row=1, col=2)

        fig.add_vline(x=strike, line_dash="dash", line_color="white", opacity=0.3, row=1, col=1)
        fig.add_vline(x=strike, line_dash="dash", line_color="white", opacity=0.3, row=1, col=2)

        fig.update_layout(template="plotly_dark", height=500, margin=dict(t=40, b=20, l=20, r=20), hovermode="x unified")
        fig.show()

w_type_am.observe(update_american_dashboard, 'value')
w_spot_am.observe(update_american_dashboard, 'value')
w_strike_am.observe(update_american_dashboard, 'value')
w_mat_am.observe(update_american_dashboard, 'value')
w_vol_am.observe(update_american_dashboard, 'value')
w_rate_am.observe(update_american_dashboard, 'value')
w_div_am.observe(update_american_dashboard, 'value')
w_steps.observe(update_american_dashboard, 'value')

display(ui_am, out_am)
update_american_dashboard()

In [ ]:
import plotly.graph_objects as go
import numpy as np

def plot_3d_american_premium(option_type='Put', K=100.0, r=0.05, q=0.0, vol=0.25, N=40):
    spots = np.linspace(60, 140, 25)
    maturities = np.linspace(0.1, 2.0, 25)

    S_grid, T_grid = np.meshgrid(spots, maturities)
    Premium_grid = np.zeros_like(S_grid)

    print(f"Calcul de la surface 3D pour un {option_type} en cours...")
    for i in range(S_grid.shape[0]):
        for j in range(S_grid.shape[1]):
            S = S_grid[i, j]
            T = T_grid[i, j]
            mkt = MarketEnvironment(S, r, q, vol)

            opt_eur = VanillaOption(K, T, option_type=option_type, is_american=False)
            p_eur = BlackScholesPricer(opt_eur, mkt).price()

            opt_ame = VanillaOption(K, T, option_type=option_type, is_american=True)
            p_ame = BinomialTreePricer(opt_ame, mkt, steps=N).price()

            Premium_grid[i, j] = max(0, round(p_ame - p_eur, 4))

    fig = go.Figure(data=[go.Surface(
        x=spots,
        y=maturities,
        z=Premium_grid,
        colorscale='Inferno',
        contours_z=dict(show=True, usecolormap=True, highlightcolor="white", project_z=True)
    )])

    fig.update_layout(
        title=f"<b>Surface 3D : Prime d'Exercice Anticipé ({option_type} Américain)</b><br><sup>Z = Prix Américain (Arbre) - Prix Européen (BSM)</sup>",
        scene=dict(
            xaxis_title='Spot Price (S)',
            yaxis_title='Maturité (T en années)',
            zaxis_title='Prime (Surcoût)',
            camera=dict(eye=dict(x=-1.5, y=-1.5, z=0.5))
        ),
        template="plotly_dark",
        height=800,
        margin=dict(l=0, r=0, b=0, t=60)
    )

    fig.show()

plot_3d_american_premium(option_type='Put', r=0.08)